# 现代购物篮分析

Polars 整理订单，FP-Growth 挖掘关联规则。

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "tutorials" / "common.py").exists():
        project_root = str(candidate)
        if project_root not in sys.path:
            sys.path.insert(0, project_root)
        break
else:
    raise FileNotFoundError("未找到项目根目录")

from tutorials.common import DATA_DIR, display

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl

try:
    from mlxtend.frequent_patterns import association_rules, fpgrowth
except ImportError as exc:
    raise ImportError("请先运行: uv sync") from exc


## 1. 构建购物篮

读取订单并透视为布尔矩阵。

In [ ]:
orders = pl.read_csv(DATA_DIR / "orders.csv", encoding="gbk")
items = pl.read_csv(DATA_DIR / "items.csv", encoding="gbk")

basket_pl = (
    orders.with_columns(pl.lit(1).alias("present"))
    .pivot(index="id", on="Goods", values="present", aggregate_function="max")
    .fill_null(0)
    .sort("id")
)
basket = pd.DataFrame(
    basket_pl.drop("id").to_numpy(),
    columns=basket_pl.columns[1:],
    index=basket_pl["id"].to_list(),
).astype(bool)
hot_goods = basket.sum().sort_values(ascending=False)

display(orders.head())
display(basket.head().astype(int))
display(hot_goods.head(10).to_frame("order_count"))


## 2. 热销概览

统计商品和品类订单数。

In [ ]:
top_goods = hot_goods.head(10)
type_counts = (
    pl.DataFrame({"Goods": hot_goods.index.to_list(), "order_count": hot_goods.to_list()})
    .join(items, on="Goods", how="left")
    .group_by("Types")
    .agg(pl.col("order_count").sum())
    .sort("order_count", descending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)
top_goods.sort_values().plot(kind="barh", ax=axes[0], color="#4C78A8")
axes[0].set_title("热销前十商品")
axes[0].set_xlabel("订单数")

axes[1].barh(
    type_counts.head(10).sort("order_count")["Types"].to_list(),
    type_counts.head(10).sort("order_count")["order_count"].to_list(),
    color="#F58518",
)
axes[1].set_title("热销前十品类")
axes[1].set_xlabel("订单数")
plt.tight_layout()
plt.show()

display(type_counts.head(10))


## 3. 关联规则

用 FP-Growth 生成规则。

In [ ]:
def make_rules(basket_frame, min_support=0.055, min_confidence=0.2):
    itemsets = fpgrowth(basket_frame, min_support=min_support, use_colnames=True)
    try:
        rules = association_rules(itemsets, metric="confidence", min_threshold=min_confidence)
    except TypeError:
        rules = association_rules(
            itemsets,
            num_itemsets=len(itemsets),
            metric="confidence",
            min_threshold=min_confidence,
        )
    return itemsets, rules


def format_rules(rules):
    selected = rules[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
    selected["antecedents"] = selected["antecedents"].apply(lambda x: ", ".join(sorted(x)))
    selected["consequents"] = selected["consequents"].apply(lambda x: ", ".join(sorted(x)))
    return selected.sort_values(["confidence", "lift"], ascending=False)


itemsets, rules = make_rules(basket, min_support=0.055, min_confidence=0.2)
display(itemsets.sort_values("support", ascending=False).head(10))
display(format_rules(rules).head(10).round(4))


## 4. 品类规则

按品类复用同一流程。

In [ ]:
orders_with_items = orders.join(items, on="Goods", how="left")
type_basket_pl = (
    orders_with_items.with_columns(pl.lit(1).alias("present"))
    .pivot(index="id", on="Types", values="present", aggregate_function="max")
    .fill_null(0)
    .sort("id")
)
type_basket = pd.DataFrame(
    type_basket_pl.drop("id").to_numpy(),
    columns=type_basket_pl.columns[1:],
    index=type_basket_pl["id"].to_list(),
).astype(bool)

type_itemsets, type_rules = make_rules(type_basket, min_support=0.25, min_confidence=0.2)

display(type_basket.head().astype(int))
display(type_itemsets.sort_values("support", ascending=False))
display(format_rules(type_rules).head(10).round(4))
